# 04 · Sentiment / Emotion Analysis

Scores each book description on 7 emotions using `j-hartmann/emotion-english-distilroberta-base`, taking the max score per emotion across all sentences in the description.

## Setup

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd

from src.config import BOOKS_WITH_CATEGORIES_CSV, EMOTION_LABELS
from src.features.emotions import (
    load_emotion_classifier, calculate_max_emotion_scores,
    compute_emotion_scores, build_books_with_emotions,
)

## 1. Load books with categories

In [ ]:
books = pd.read_csv(BOOKS_WITH_CATEGORIES_CSV)

## 2. Load the emotion classifier

Set `device=0` below if you have a CUDA GPU available.

In [ ]:
classifier = load_emotion_classifier(device=-1)
classifier("messi goaaaaaaal")

### Quick look at one description's sentence-level predictions

In [ ]:
sentences = books["description"][0].split(".")
predictions = classifier(sentences)
predictions[0]

## 3. Score emotions for every book

This can take a while on CPU — it runs the classifier over every sentence in every book's description.

In [ ]:
emotions_df = compute_emotion_scores(books, classifier)
emotions_df.head()

## 4. Merge & save

(equivalent to calling `build_books_with_emotions()` directly — shown step by step above for exploration purposes)

In [ ]:
from src.config import BOOKS_WITH_EMOTIONS_CSV
books = pd.merge(books, emotions_df, on="isbn13")
books.to_csv(BOOKS_WITH_EMOTIONS_CSV, index=False)
print("Saved:", books.shape, "->", BOOKS_WITH_EMOTIONS_CSV)

---
All steps above are also available end-to-end via `python -m src.pipeline`. Launch the app with `python -m src.app.dashboard`.